# 01. Specialized Language Model for Low-Resource African Language (Ewe / Èʋegbe)

**Course**: ICS554 Natural Language Processing · Ashesi University  
**Project**: Prosit 1 (Ankora AI Research Lab)  
**Target Language**: **Ewe (Èʋegbe)** — spoken in Ghana, Togo, and Benin  
**Objective**: Develop and evaluate a statistical n-gram language model for Ewe, addressing data scarcity, unique orthographic characters (`ɖ`, `ƒ`, `ɣ`, `ŋ`, `ɔ`, `ɛ`, `ʋ`), out-of-vocabulary words (`<unk>`), and smoothing techniques.

---
### Pipeline Overview
1. **Data Ingestion**: Load Ewe text from a local file (`data/raw/low_resource/`) or Hugging Face dataset.
2. **Ewe Unicode Tokenization**: NFC normalization to preserve tone diacritics and distinct Ewe alphabetic characters.
3. **Sentence Boundaries & OOV Protocol**: Prepend `<s>` to evaluate initial token conditional probability $P(w_1 \mid \text{<s>})$, append `</s>`, and replace rare words with `<unk>` strictly using training frequencies.
4. **Model Training**: Unigram, Bigram, and Trigram count estimation.
5. **Smoothing & Evaluation**: Compare MLE, Laplace, Lidstone, Linear Interpolation, and Interpolated Kneser-Ney via Perplexity (PP).

In [ ]:
import sys
import random
from pathlib import Path

# Ensure repo root is on python path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import (
    basic_tokenize,
    build_vocabulary,
    replace_oov_tokens,
    load_corpus_from_file_or_hf,
)
from src.ngram import NGramLM
from src.viz import plot_ngram_frequency, plot_perplexity_comparison

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

## 1. Load Ewe Corpus
You can drop your Ewe text file into `data/raw/low_resource/ewe.txt` (or provide a Hugging Face dataset ID).
If the file is not yet placed, the pipeline automatically falls back to curated multi-domain Ewe sentences for seamless local execution.

In [ ]:
LOCAL_EWE_PATH = REPO_ROOT / "data" / "raw" / "low_resource" / "ewe.txt"

# Attempt loading from local path first, otherwise fallback to curated baseline
corpus_lines = load_corpus_from_file_or_hf(str(LOCAL_EWE_PATH))

if not corpus_lines:
    print("Local dataset not found yet. Using representative curated Ewe sentences...")
    corpus_lines = [
        "Woezɔ loo, miawo katã míedi ŋutifafa le dukɔa me",
        "Efoa nyuie mah? Nyee, mefo nyuie, akpe kaka",
        "Kofi yi suku le Keta egbe ŋdi",
        "Ama fle nuɖuɖu vivi le asime le Ho",
        "Míeyi aƒeme kaba elabena tsi le dzadzam le Aflao",
        "Nufiala fia nu nusrɔ̃lawo nyuie le suku me",
        "Mia dogo le etsɔ me ne Mawu lɔ̃",
        "Devi sia nya nu ŋutɔ le eƒe nusɔsrɔ̃ me",
        "Míedi be míawɔ dɔ le ɖekawɔwɔ me",
        "Ɖo to nyuie ne nàse nya si gblɔm wole",
        "Agbledeŋu nye dɔ vevi aɖe le miaƒe nutoa me",
        "Míele kuku ɖem na mi be miagbɔ kaba",
        "Ŋutsu la kple nyɔnu la woyi agble me",
        "Mía kplɔlawo le dɔ wɔm be dukɔa nade ŋgɔ",
        "Akpe na mi katã ɖe miaƒe kpekpeɖeŋu ta"
    ]

print(f"Total Ewe sentences ready for processing: {len(corpus_lines)}")

## 2. Unicode Tokenization & Leak-Free Train/Test Split
Ewe contains distinctive phonemes and glyphs (`ɖ`, `ƒ`, `ɣ`, `ŋ`, `ɔ`, `ɛ`, `ʋ`). Our tokenizer uses Unicode NFC normalization to preserve tone markers and prevent morphological fragmentation.

In [ ]:
# Tokenize preserving Ewe characters
tokenized_data = [basic_tokenize(line) for line in corpus_lines if line.strip()]

# 80/20 train/test split
split_idx = max(1, int(0.8 * len(tokenized_data)))
train_tokens = tokenized_data[:split_idx]
test_tokens = tokenized_data[split_idx:]

# Induce closed vocabulary STRICTLY from training partition to prevent leakage
vocab, freqs = build_vocabulary(train_tokens, min_freq=1)
train_clean = replace_oov_tokens(train_tokens, vocab)
test_clean = replace_oov_tokens(test_tokens, vocab)

print(f"Vocabulary size: {len(vocab)} unique tokens")
print(f"Training sequences: {len(train_clean)}, Test sequences: {len(test_clean)}")

## 3. Training Ewe N-Gram Models with Smoothing
We train statistical models with prepended `<s>` start tokens to evaluate conditional probabilities: $P(w_1 \mid \text{<s>})$.

In [ ]:
# 1. Unigram with Laplace
unigram = NGramLM(n=1, smoothing="laplace").fit(train_clean, vocab=vocab)

# 2. Bigram with Laplace (Add-One)
bigram_laplace = NGramLM(n=2, smoothing="laplace", k=1.0).fit(train_clean, vocab=vocab)

# 3. Bigram with Lidstone (Add-0.1)
bigram_lidstone = NGramLM(n=2, smoothing="laplace", k=0.1).fit(train_clean, vocab=vocab)

# 4. Trigram with Linear Interpolation
trigram_interp = NGramLM(n=3, smoothing="interpolation").fit(train_clean, vocab=vocab)
trigram_interp.set_interpolation_weights([0.1, 0.3, 0.6])

# 5. Bigram with Interpolated Kneser-Ney (Continuation Probabilities)
bigram_kn = NGramLM(n=2, smoothing="kneser_ney").fit(train_clean, vocab=vocab)

print("All Ewe n-gram models trained successfully.")

## 4. Intrinsic Evaluation: Perplexity on Unseen Ewe Test Split

In [ ]:
models = {
    "Unigram (Laplace)": unigram,
    "Bigram (Laplace)": bigram_laplace,
    "Bigram (Add-0.1)": bigram_lidstone,
    "Trigram (Interpolation)": trigram_interp,
    "Bigram (Kneser-Ney)": bigram_kn,
}

results = {}
for name, model in models.items():
    ppl = model.perplexity(test_clean)
    results[name] = ppl
    print(f"{name:25s} -> Perplexity: {ppl:.2f}")

# Plot comparison
plot_perplexity_comparison(
    list(results.keys()),
    list(results.values()),
    title="Ewe Language Model - Test Perplexity Benchmark",
    save_path=REPO_ROOT / "figures" / "ngram_perplexity_comparison.png",
)

## 5. Text Generation in Ewe with Sampling Temperature

In [ ]:
print("--- Generated Ewe Text Samples ---")
for name, model in models.items():
    sample = model.generate(max_length=10, temperature=0.7)
    print(f"[{name}]: {sample}")